# Instalacao das bibliotecas


In [1]:
!pip install numpy==1.23.5 pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 27.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
db-dtypes 1.4.3 requires numpy>=1.24.0, but you have numpy 1.23.5 which is incompatible.
albucore 0.0.24 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
albumentations 2.0.8 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.23.5 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
bigfra

In [1]:
!pip install pmdarima

In [2]:
!pip install deap

# Importacao das bibliotecas e dos dados


In [3]:
import pandas as pd
import os
import numpy as np

In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from deap import base, creator, tools, algorithms
import random

In [5]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import joblib

In [6]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:
caminho_dados = '/content/drive/Shareddrives/IC - Otimizacao computacional para inflacao/Dados/dados_USA'

# Listar os arquivos dentro da pasta "Dados"
arquivos_dados = os.listdir(caminho_dados)

print(arquivos_dados)


['Untitled0.ipynb', 'dados_macroeconomicos_usa.csv']


In [10]:
# Caminho para o arquivo CSV
caminho_arquivo = os.path.join(caminho_dados, 'dados_macroeconomicos_usa.csv')

# Ler o arquivo CSV
df = pd.read_csv(caminho_arquivo)

In [11]:
df = df.set_index('DATE')
df.drop(columns=['Unnamed: 0'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180 entries, 2010-01-01 to 2024-12-01
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Inflacao_CPI                 180 non-null    float64
 1   Inflacao_CPI_Core            180 non-null    float64
 2   PCE_Core                     180 non-null    float64
 3   Desemprego                   180 non-null    float64
 4   Desemprego_menos_27_semanas  180 non-null    float64
 5   Taxa_Juros_Fed               180 non-null    float64
 6   Emprego_Total_Nao_Agricola   180 non-null    int64  
 7   Permissoes_Construcao        180 non-null    int64  
 8   Salario_Medio                180 non-null    float64
 9   Salario_Medio_Real_Hora      180 non-null    float64
 10  Producao_Industrial          180 non-null    float64
 11  Capacidade_Instalada         180 non-null    float64
 12  PCE                          180 non-null    float64
 13  Deflator_

# Divisao dos dados e treinamento

In [12]:
# Divisao do alvo e dados de treino
X = df.drop(columns = ['Inflacao_CPI'])
y = df['Inflacao_CPI']

In [13]:
# Normalizacao dos dados usando MinMax
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)

# Split dos dados com janela de 5
tscv = TimeSeriesSplit(n_splits=5)

# Criacao da funcao fitness

In [14]:
def feature_selection_fitness(individual, model, X_scaled, y, tscv, metric_func, max_features_allowed=None):
    """
    Parâmetros:
    - individual: cromossomo binário representando as features selecionadas
    - model: modelo de aprendizado
    - X_scaled: dados de entrada escalados (pandas DataFrame)
    - y: variável alvo (pandas Series)
    - tscv: TimeSeriesSplit
    - metric_func: função para calcular a métrica
    - max_features_allowed: número máximo de features permitidas (int ou None)

    Retorna:
    - métrica de avaliação para o conjunto de features selecionadas
    """

    # Garantir que X_scaled seja um DataFrame
    if isinstance(X_scaled, np.ndarray):
        X_scaled = pd.DataFrame(X_scaled, columns=[f"feature_{i}" for i in range(X_scaled.shape[1])])

    selected_features = [feature for i, feature in enumerate(X_scaled.columns) if individual[i] == 1]

    # Penalizar se não houver features ou se exceder o máximo permitido
    if len(selected_features) == 0:
        return -9999.,
    if max_features_allowed is not None and len(selected_features) > max_features_allowed:
        return -9999.,

    # Realiza a divisão temporal dos dados
    train_index, test_index = list(tscv.split(X_scaled))[-1]

    X_train, X_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    try:
        model.fit(X_train[selected_features], y_train)
        y_pred = model.predict(X_test[selected_features])
        score = metric_func(y_test, y_pred)
    except Exception as e:
        print(f"Erro ao treinar ou avaliar o modelo: {e}")
        return -9999.,

    return score,


Funcoes auxiliares

# Funcoes auxiliares

In [15]:
#Funcoes auxiliares

def make_stationary(df, max_diff=5):
    """Torna a série temporal estacionária através de diferenciação."""
    df_diff = df.copy()
    for i in range(1, max_diff + 1):
        stationary = True
        for column in df_diff.columns:
            if not check_stationarity(df_diff[[column]]):
                stationary = False
                df_diff[[column]] = df_diff[[column]].diff().fillna(method='ffill').fillna(method='bfill')
        if stationary:
            print(f"Série estacionária após {i-1} diferenciações.")
            return df_diff.dropna()
    raise ValueError("A série não se tornou estacionária após o número máximo de diferenciações.")

def check_stationarity(series, significance_level=0.05):
    """Verifica se a série é estacionária usando o teste ADF."""
    from statsmodels.tsa.stattools import adfuller
    result = adfuller(series.dropna())
    return result[1] < significance_level

def plot_forecasts(y_test, final_forecast, title='Previsão Dinâmica'):
    """Plota as previsões comparadas com os valores reais."""
    plt.figure(figsize=(15, 3))
    plt.plot(y_test.index, y_test, label='Valores Reais', color='blue')
    plt.plot(y_test.index, final_forecast, label='Previsão', color='orange')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# Hibrido (ARIMAX+RF) MODEL

In [16]:
def executar_modelo_arimax_hibrido(df, exogenous_vars):
    # Tornar o DataFrame estacionário
    df_stationary = make_stationary(df[['Inflacao_CPI'] + exogenous_vars])

    # Dividir em treino e teste
    train_size = int(len(df_stationary) * 0.8)
    train, test = df_stationary.iloc[:train_size], df_stationary.iloc[train_size:]

    # Variável dependente e exógenas
    y_train = train['Inflacao_CPI']
    y_test = test['Inflacao_CPI']
    X_train = train[exogenous_vars]
    X_test = test[exogenous_vars]

    # Ajustar modelo ARIMAX com auto_arima
    arimax_model = auto_arima(y_train, exogenous=X_train, seasonal=False, trace=True,
                               error_action='ignore', suppress_warnings=True)
    arimax_order = arimax_model.order

    # Previsão dinâmica com atualização a cada passo
    dynamic_arimax_forecast = []
    history_y = list(y_train)
    history_X = list(X_train.values)

    for t in range(len(y_test)):
        model = SARIMAX(history_y, exog=np.array(history_X), order=arimax_order)
        fitted_model = model.fit(disp=False)
        next_x = X_test.iloc[t].values.reshape(1, -1)
        forecast = fitted_model.forecast(steps=1, exog=next_x)
        dynamic_arimax_forecast.append(forecast[0])
        history_y.append(y_test.iloc[t])
        history_X.append(X_test.iloc[t].values)

    # Calcular resíduos
    residuals = y_test.values - dynamic_arimax_forecast

    # Treinar modelo Random Forest para prever os resíduos
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_test.reset_index(drop=True), residuals)

    # Previsão dos resíduos com Random Forest
    predicted_residuals_rf = rf_model.predict(X_test.reset_index(drop=True))

    # Previsão híbrida: ARIMAX + RF
    hybrid_forecast = np.array(dynamic_arimax_forecast) + predicted_residuals_rf

    # Métricas de desempenho
    mse = mean_squared_error(y_test, dynamic_arimax_forecast)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, dynamic_arimax_forecast)
    mape = np.mean(np.abs((y_test - dynamic_arimax_forecast) / y_test))

    return mae
    '''
    # Resultados
    print(f'MAE: {mae:.6f}')
    print(f'MSE: {mse:.6f}')
    print(f'RMSE: {rmse:.6f}')
    print(f'MAPE: {mape:.6f}')
    print(f'R²: {r2:.6f}')

    # Plotar resultados
    plot_forecasts(y_test, hybrid_forecast, title=f'{tecnica_nome} (ARIMAX + Random Forest)')
    '''

# Funcao fitness hibrido

In [17]:
def arimax_feature_fitness_hibrido(individual, df, all_features):
    """
    Calcula o fitness do indivíduo treinando e avaliando um modelo ARIMAX + Random Forest híbrido.
    """
    # Seleciona as features ativas no indivíduo
    selected_features = [feature for gene, feature in zip(individual, all_features) if gene == 1]

    if len(selected_features) == 0:
        return (9999.,)  # Penalidade para indivíduos sem features

    try:
        # Executa o modelo híbrido e calcula o MAE
        mae = executar_modelo_arimax_hibrido(df.copy(), selected_features)

        print("\nAvaliação do Modelo Atual:")
        print("Features Exógenas Utilizadas:", selected_features)
        print("MAE (Erro Médio Absoluto):", mae)

    except Exception as e:
        print(f"Erro no ajuste ou predição do modelo híbrido: {e}")
        return (9999.,)  # Penalidade forte em caso de falha

    return (mae,)


# Implementacao da selecao

In [18]:
import warnings
warnings.filterwarnings('ignore')


In [19]:
from deap import base, creator, tools, algorithms
import random

# Lista de todas as variáveis exógenas disponíveis
ALL_FEATURES = df.columns.drop('Inflacao_CPI').tolist()

# Definindo o número máximo de features permitidas
MAX_FEATURES_ALLOWED = 5  # <-- Ajuste aqui o limite desejado

# Criando as classes de fitness e indivíduo
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Função para criar um indivíduo respeitando o limite de features
def create_individual():
    individual = [0] * len(ALL_FEATURES)
    selected_indices = random.sample(range(len(ALL_FEATURES)), MAX_FEATURES_ALLOWED)
    for idx in selected_indices:
        individual[idx] = 1
    return creator.Individual(individual)

# População
toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Função de avaliação (fitness) usando ARIMAX
toolbox.register("evaluate", lambda ind: arimax_feature_fitness_hibrido(ind, df, ALL_FEATURES))

# Cruzamento (não precisa mudar)
toolbox.register("mate", tools.cxTwoPoint)

# Nova função de mutação que respeita o limite
def mutate(individual, indpb):
    for i in range(len(individual)):
        if random.random() < indpb:
            if individual[i] == 1:
                individual[i] = 0
            else:
                # Só liga a feature se ainda não atingiu o máximo
                if sum(individual) < MAX_FEATURES_ALLOWED:
                    individual[i] = 1
    # Se por acaso zerar todas, força pelo menos 1 ativa
    if sum(individual) == 0:
        idx = random.randrange(len(individual))
        individual[idx] = 1
    return individual,

toolbox.register("mutate", mutate, indpb=0.2)

# Seleção
toolbox.register("select", tools.selTournament, tournsize=3)

In [20]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Parâmetros do GA
population_size = 10
generations = 5
crossover_prob = 0.7
mutation_prob = 0.2

# População inicial
population = toolbox.population(n=population_size)

# ====== Loop do GA manual para controlar impressões ======

for gen in range(generations):
    print(f"\n--- Geração {gen+1} ---")

    # Cruzamento
    offspring = algorithms.varAnd(population, toolbox, cxpb=crossover_prob, mutpb=mutation_prob)

    # Avaliar indivíduos que ainda não possuem fitness calculado
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]

    for ind in invalid_ind:
        # Avalia e já imprime os exógenos usados
        fitness = toolbox.evaluate(ind)
        ind.fitness.values = fitness

    # Substituir população
    population[:] = toolbox.select(offspring, k=len(population))

    # Melhor indivíduo da geração
    best_ind = tools.selBest(population, k=1)[0]
    selected_features = [f for gene, f in zip(best_ind, ALL_FEATURES) if gene == 1]

    print("Melhores features nesta geração:", selected_features)
    print("Fitness:", best_ind.fitness.values[0])

# ====== Fim do GA ======

# Melhor solução final
best_ind = tools.selBest(population, k=1)[0]
selected_features = [f for gene, f in zip(best_ind, ALL_FEATURES) if gene == 1]

print("\n=== Melhor conjunto de features encontrado ===")
print("Features:", selected_features)
print("Fitness final:", best_ind.fitness.values[0])



--- Geração 1 ---
Série estacionária após 2 diferenciações.
Performing stepwise search to minimize aic
 ARIMA(2,0,2)(0,0,0)[0]             : AIC=274.464, Time=0.19 sec
 ARIMA(0,0,0)(0,0,0)[0]             : AIC=312.107, Time=0.04 sec
 ARIMA(1,0,0)(0,0,0)[0]             : AIC=314.050, Time=0.04 sec
 ARIMA(0,0,1)(0,0,0)[0]             : AIC=313.963, Time=0.04 sec
 ARIMA(1,0,2)(0,0,0)[0]             : AIC=275.658, Time=0.13 sec
 ARIMA(2,0,1)(0,0,0)[0]             : AIC=272.474, Time=0.11 sec
 ARIMA(1,0,1)(0,0,0)[0]             : AIC=285.076, Time=0.12 sec
 ARIMA(2,0,0)(0,0,0)[0]             : AIC=300.652, Time=0.04 sec
 ARIMA(3,0,1)(0,0,0)[0]             : AIC=274.459, Time=0.16 sec
 ARIMA(3,0,0)(0,0,0)[0]             : AIC=290.889, Time=0.06 sec
 ARIMA(3,0,2)(0,0,0)[0]             : AIC=276.391, Time=0.30 sec
 ARIMA(2,0,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.39 sec

Best model:  ARIMA(2,0,1)(0,0,0)[0]          
Total fit time: 1.646 seconds

Avaliação do Modelo Atual:
Features Exógen